In [ ]:
ReportFolderName = 'GPT-Based_Base_Version'
model_alias = "Gemma3"
random_state = 43  # unified with BERT notebook for fair cross-model comparison


In [ ]:
import shutil
import os

# Keywords for folders to delete
folders_to_delete = ["logs", ReportFolderName, "results", "sample_data"]

# Delete matching folders
for item in os.listdir("."):
    if os.path.isdir(item) and any(keyword in item for keyword in folders_to_delete):
        shutil.rmtree(item)
        print(f"✅ Deleted folder: {item}")

# Delete all files in the current directory
for item in os.listdir("."):
    if os.path.isfile(item):
        os.remove(item)
        print(f"🗑️ Deleted file: {item}")

print("\n🎯 Full cleanup completed. All matching folders and all files removed.")

✅ Deleted folder: sample_data

🎯 Full cleanup completed. All matching folders and all files removed.


In [ ]:
reports_dir = f"{ReportFolderName}"
os.makedirs(reports_dir, exist_ok=True)

In [ ]:
!pip install -q -U transformers datasets peft accelerate bitsandbytes trl
!pip install -q scikit-learn pandas numpy tqdm "torchao>=0.16.0"

import os
os.environ["WANDB_MODE"] = "disabled"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 141.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 47.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 39.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 42.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 825.1/825.1 kB 62.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 52.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 104.4 MB/s eta 0:00:00


In [ ]:

# ============================================
# STEP 1: INSTALL DEPENDENCIES
# ============================================
# Uncomment these lines if running in a new Colab environment
# !pip install -q -U transformers datasets peft accelerate bitsandbytes trl
# !pip install -q scikit-learn pandas numpy tqdm

import torch
import numpy as np
import pandas as pd
from datasets import load_dataset, DatasetDict,Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from tqdm import tqdm

# ============================================
# STEP 2: CONFIGURATION
# ============================================
MODEL_ID = "google/gemma-3-4b-it"
VALID_LABELS = ["smish", "promo", "normal"]

In [ ]:
# ============================================
# STEP 3: LOAD DATA & PREPARE SPLITS
# ============================================
print("\n📥 Loading SMS dataset...")
dataset = load_dataset("shariul-islam/bengali-sms-smishing-dataset")

# Only for two variant
variants = ["Bengali", "English"]
train_dataset_filtered = dataset["train"].filter(lambda example: example["source"] in variants)
dataset = DatasetDict({
    "train": train_dataset_filtered,
    "validation": dataset["validation"],
    "test": dataset["test"]
})

# Shuffle and select samples
all_data = dataset['train'].shuffle(seed=42)

train_samples = dataset['train']
test_samples = dataset['test']
validation_samples = dataset['validation']

# Select Train (100) and Test (20) samples
# train_samples = all_data.select(range(100))
# test_samples = all_data.select(range(100, 120))
# validation_samples = all_data.select(range(100, 110))


print(f"✅ Train samples: {len(train_samples)}")
print(f"✅ Test samples: {len(test_samples)}")
print(f"✅ validation samples: {len(validation_samples)}")



📥 Loading SMS dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md:   0%|          | 0.00/3.03k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/404k [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/60.6k [00:00<?, ?B/s]

...

Generating train split:   0%|          | 0/4903 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/701 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1401 [00:00<?, ? examples/s]

Filter:   0%|          | 0/4903 [00:00<?, ? examples/s]

✅ Train samples: 2467
✅ Test samples: 1401
✅ validation samples: 701


In [ ]:
# ============================================
# STEP 3: LOAD DATA & PREPARE SPLITS
# ============================================
print("\n📥 Loading SMS dataset...")
dataset = load_dataset("shariul-islam/bengali-sms-smishing-dataset")

# Only for two variant
variants = ["Bengali", "English"]
train_dataset_filtered = dataset["train"].filter(lambda example: example["source"] in variants)
validation_dataset_filtered = dataset["validation"].filter(lambda example: example["source"] in variants)
dataset = DatasetDict({
    "train": train_dataset_filtered,
    "validation": validation_dataset_filtered,
    "test": dataset["test"]
})

# Shuffle and select samples
all_data = dataset['train'].shuffle(seed=42)

train_samples = dataset['train']
test_samples = dataset['test']
validation_samples = dataset['validation']

# Select Train (100) and Test (20) samples
# train_samples = all_data.select(range(100))
# test_samples = all_data.select(range(100, 120))
# validation_samples = all_data.select(range(100, 110))


print(f"✅ Train samples: {len(train_samples)}")
print(f"✅ Test samples: {len(test_samples)}")
print(f"✅ validation samples: {len(validation_samples)}")



📥 Loading SMS dataset...


Filter:   0%|          | 0/701 [00:00<?, ? examples/s]

✅ Train samples: 2467
✅ Test samples: 1401
✅ validation samples: 353


In [ ]:
# ============================================
# STEP 3: LOAD DATA & PREPARE SPLITS
# ============================================
print("\n📥 Loading SMS dataset...")
dataset = load_dataset("shariul-islam/bengali-sms-smishing-dataset")

# Only for two variant
variants = ["Bengali", "English"]
train_dataset_filtered = dataset["train"].filter(lambda example: example["source"] in variants)
validation_dataset_filtered = dataset["validation"].filter(lambda example: example["source"] in variants)
dataset = DatasetDict({
    "train": train_dataset_filtered,
    "validation": validation_dataset_filtered,
    "test": dataset["test"]
})

# Shuffle and select samples
all_data = dataset['train'].shuffle(seed=42)

train_samples = dataset['train']
test_samples = dataset['test']
validation_samples = dataset['validation']

# Select Train (100) and Test (20) samples
# train_samples = all_data.select(range(100))
# test_samples = all_data.select(range(100, 120))
# validation_samples = all_data.select(range(100, 110))


print(f"✅ Train samples: {len(train_samples)}")
print(f"✅ Test samples: {len(test_samples)}")
print(f"✅ validation samples: {len(validation_samples)}")



📥 Loading SMS dataset...
✅ Train samples: 2467
✅ Test samples: 1401
✅ validation samples: 353


In [ ]:

# ============================================
# STEP 4: PROMPT TEMPLATES & HELPERS
# ============================================
def get_zero_shot_prompt(sms: str) -> str:
    return f'''You are an expert in SMS content classification for fraud detection and marketing analysis.

SMS: "{sms}"

Task: Classify the above SMS message into one of three categories:
1. smish — Fraudulent or scam SMS that tries to trick users into revealing sensitive information, clicking malicious links, or calling scam numbers.
2. promo — Promotional or marketing SMS offering discounts, sales, cashback, or advertisements.
3. normal — Regular personal messages, greetings, casual conversations.

Instructions:
- Response will only be either smish, promo, or normal.
- A single-word response.

Response:'''

def get_training_prompt(sms: str, label: str) -> str:
    return f"{get_zero_shot_prompt(sms)} {label}"

def prepare_training_data(samples) -> Dataset:
    texts = []
    for sample in samples:
        prompt = get_training_prompt(sample['text'], sample['label'])
        texts.append({"text": prompt})
    return Dataset.from_list(texts)

def classify_sms(model, tokenizer, sms_text: str) -> str:
    """Classify a single SMS message"""
    prompt = get_zero_shot_prompt(sms_text)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=10,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.2,
        )

    generated = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    response = generated.strip().lower()

    # Simple mapping to handle extra punctuation
    if 'smish' in response: return 'smish'
    if 'promo' in response: return 'promo'
    if 'normal' in response: return 'normal'
    return "unknown"

def classify_sms_with_probs(model, tokenizer, sms_text: str) -> tuple:
    """Classify SMS with probability scores for each class"""
    prompt = get_zero_shot_prompt(sms_text)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # Get the token IDs for our labels.
    # IMPORTANT: after "Response:" the model predicts a SPACE-PREFIXED token,
    # so we encode " smish"/" promo"/" normal" (leading space) to read the
    # correct logits. Encoding without the space measured the wrong token.
    smish_token = tokenizer.encode(" smish", add_special_tokens=False)[0]
    promo_token = tokenizer.encode(" promo", add_special_tokens=False)[0]
    normal_token = tokenizer.encode(" normal", add_special_tokens=False)[0]

    with torch.no_grad():
        # Get logits instead of generating
        outputs = model(**inputs)

        # Get logits for the next token (last position)
        next_token_logits = outputs.logits[0, -1, :]

        # Apply softmax to get probabilities
        probs = torch.softmax(next_token_logits, dim=-1)

        # Extract probabilities for our target labels
        prob_smish = probs[smish_token].item()
        prob_promo = probs[promo_token].item()
        prob_normal = probs[normal_token].item()

        # Normalize to sum to 1 (since we only care about these 3)
        total = prob_smish + prob_promo + prob_normal
        prob_smish_norm = prob_smish / total
        prob_promo_norm = prob_promo / total
        prob_normal_norm = prob_normal / total

        # Get predicted class
        probs_dict = {
            'smish': prob_smish_norm,
            'promo': prob_promo_norm,
            'normal': prob_normal_norm
        }
        predicted = max(probs_dict, key=probs_dict.get)

    return predicted, probs_dict



In [ ]:
# ============================================
# STEP 5: LOAD BASE MODEL (QUANTIZED)
# ============================================
print("\n⚙️ Loading Base Model (4-bit)...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.bfloat16, # Explicitly set this
    trust_remote_code=True,
)


⚙️ Loading Base Model (4-bit)...


OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/google/gemma-3-4b-it.
401 Client Error. (Request ID: Root=1-6a2e7fb7-4a250bf351db999b4231ee98;a45fb0af-921f-46a4-96c7-da249db1fe32)

Cannot access gated repo for url https://huggingface.co/google/gemma-3-4b-it/resolve/main/config.json.
Access to model google/gemma-3-4b-it is restricted. You must have access to it and be authenticated to access it. Please log in.

In [ ]:
# ============================================
# STEP 7: PREPARE FOR FINE-TUNING
# ============================================
print("\n🔧 Preparing model for LoRA training...")
base_model = prepare_model_for_kbit_training(base_model)


def preprocess_logits_for_metrics(logits, labels):
    """Collapse full-vocabulary logits to argmax token-ids DURING eval.
    Without this, the Trainer accumulates (batch, seq, vocab~256k) float logits
    across the whole eval set and OOMs. We only need the predicted token id."""
    if isinstance(logits, tuple):
        logits = logits[0]
    return logits.argmax(dim=-1)   # (batch, seq) int ids, not (batch, seq, vocab) floats


def compute_metrics(eval_pred):
    """Classification-only metrics for the generative SFT setup.
    Predictions arrive already argmax'd (see preprocess_logits_for_metrics)."""
    predictions, labels = eval_pred   # predictions: (batch, seq) token ids

    last_token_preds = []
    last_token_labels = []
    for i in range(labels.shape[0]):
        valid_indices = np.where(labels[i] != -100)[0]
        if len(valid_indices) > 0:
            last_idx = valid_indices[-1]
            # Logits at [last_idx - 1] predict the label token at [last_idx]
            last_token_preds.append(predictions[i, last_idx - 1])
            last_token_labels.append(labels[i, last_idx])

    acc = accuracy_score(last_token_labels, last_token_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        last_token_labels, last_token_preds, average='weighted', zero_division=0
    )
    return {
        'eval_accuracy': acc,
        'eval_f1': f1,
        'eval_precision': precision,
        'eval_recall': recall,
    }


# LoRA configuration — matches Table 3.4
# r=8, alpha=32 (scaling 4.0); Gemma-3 target modules = Q,K,V,Output,Gate,Up,Down
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",   # decoder model: generative classification (see thesis note on Table 3.4)
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

train_dataset = prepare_training_data(train_samples)
val_dataset = prepare_training_data(validation_samples)


In [ ]:
# ============================================
# SANITY CHECK — run BEFORE the full 10-epoch train
# ============================================
# 1) Confirm the three label tokens are distinct (else generative
#    classification probabilities are meaningless).
for w in ["smish", "promo", "normal"]:
    no_sp = tokenizer.encode(w, add_special_tokens=False)
    sp    = tokenizer.encode(" " + w, add_special_tokens=False)
    print(f"{w:8} | no-space first id: {no_sp[0]:>7} | space-prefixed first id: {sp[0]:>7}")

space_ids = [tokenizer.encode(" " + w, add_special_tokens=False)[0] for w in ["smish","promo","normal"]]
assert len(set(space_ids)) == 3, "❌ Label first-token ids collide — fix label words/prompt before training."
print("✅ Label tokens are distinct.")

# 2) OPTIONAL smoke test: set SMOKE_TEST=True to train ONE epoch and confirm
#    eval_f1 prints a sensible (non-zero, non-NaN) value before the full run.
SMOKE_TEST = True
if SMOKE_TEST:
    print("\n⚠️ SMOKE TEST: training 1 epoch to verify eval_f1 wiring...")


In [ ]:
# ============================================
# STEP 8: TRAIN (FINE-TUNE) — matches Table 3.6
# ============================================
from transformers import EarlyStoppingCallback

print("\n🚀 Starting LoRA Fine-Tuning (config matches Table 3.6)...")

# Effective batch size 32 = per_device 8 x grad_accum 4 (fits 4-bit Gemma-3-4B).
# If you hit OOM, set per_device_train_batch_size=4 and gradient_accumulation_steps=8.
training_args = SFTConfig(
    output_dir=f"./results_{model_alias}",
    num_train_epochs=10,                     # Table 3.6: Maximum Epochs
    per_device_train_batch_size=8,           # 8 x 4 = effective batch 32
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,           # Table 3.6: grad accumulation (Gemma)
    learning_rate=2e-4,                      # Table 3.6: Gemma-3 LoRA LR
    weight_decay=0.01,                       # Table 3.6: weight decay
    lr_scheduler_type="linear",              # Table 3.6: linear decay with warmup
    warmup_ratio=0.1,                        # Table 3.6: warmup ratio
    bf16=True,                               # Table 3.6: BF16 for Gemma-3
    max_grad_norm=0.3,
    optim="paged_adamw_8bit",                # AdamW (Table 3.6); paged 8-bit for VRAM
    # NOTE: prompt+SMS exceeds 128 tokens, so 128 (Table 3.5) cannot apply to Gemma.
    # Amend Table 3.5 to state 128 is for the encoder models' direct SMS input;
    # Gemma uses the longer instruction-prompt length below.
    max_length=512,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=20,
    load_best_model_at_end=True,             # keep best-by-eval_f1 checkpoint
    metric_for_best_model="eval_f1",         # Table 3.6: "Best weighted F1"
    greater_is_better=True,
    save_total_limit=2,
    seed=random_state,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    args=training_args,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    preprocess_logits_for_metrics=preprocess_logits_for_metrics,  # FIX: prevents eval-logits OOM
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)], # Table 3.6: patience=3
)

trainer.train()
print("\n💾 Training complete.")


In [ ]:
trainer.model.save_pretrained("./gemma3-2v-fine-tune-adapter")
trainer.model.push_to_hub(
    repo_id="shariul-islam/gemma3-2v-fine-tune-adapter",
    private=False  # set True if you want private
)


In [ ]:
# Assuming you have already loaded your tokenizer
# tokenizer = AutoTokenizer.from_pretrained("google/gemma-2b")

# Encode the prompt to token IDs
prompt = get_training_prompt('How are you kjn klkl kll lml,m lml lmlmm', 'normal')
print(prompt)
tokens = tokenizer.encode(prompt, add_special_tokens=True)

# Count the tokens
token_count = len(tokens)

print(f"The prompt contains {token_count} tokens.")

In [ ]:
# ============================================
# STEP 9: EVALUATE FINE-TUNED MODEL
# ============================================
print("\n" + "="*50)
print("🔍 EVALUATION 2: FINE-TUNED MODEL")
print("="*50)
print("Running inference on test_samples AFTER training...")

# Note: 'model' is now the Fine-Tuned model (Base + LoRA)
y_true = [sample['label'] for sample in test_samples]
y_pred = []
analysis_results = []
for sample in tqdm(test_samples, desc="Fine-Tuned Inference"):
    pred, probs = classify_sms_with_probs(model, tokenizer, sample['text'])
    y_pred.append(pred)

    # Store all relevant info in a dictionary
    analysis_results.append({
        "SMS_Text": sample['text'],
        "True_Label": sample['label'],
        "Predicted_Label": pred,
        "Prob_Smish": probs.get('smish', 0),
        "Prob_Promo": probs.get('promo', 0),
        "Prob_Normal": probs.get('normal', 0),
        "Source": sample['source'],
        "Is_Correct": 1 if pred == sample['label'] else 0
    })


# Create DataFrame
df_analysis = pd.DataFrame(analysis_results)

# Save to CSV
csv_filename = "smish_detection_fine_tuned_results.csv"
df_analysis.to_csv(f"{reports_dir}/{csv_filename}", index=False, encoding='utf-8-sig')

# Store fine-tuned metrics
acc_ft = accuracy_score(y_true, y_pred)
prec_ft, rec_ft, f1_ft, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted', zero_division=0)

print(f"\nFine-Tuned Accuracy: {acc_ft:.4f}")
print("Fine-Tuned Classification Report:")
print(classification_report(y_true, y_pred, zero_division=0))


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_curve,
    auc, precision_recall_fscore_support
)
from sklearn.preprocessing import label_binarize
import os

# --- Preparation for Metrics ---
labels = ['normal', 'promo', 'smish']
n_classes = len(labels)

# 1. GENERATE CLASSIFICATION REPORTS
report_dict = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
report_df =  pd.DataFrame(report_dict).transpose().round(4)


# Text file
report_text = classification_report(y_true, y_pred)
with open(f"{reports_dir}/classification_report.txt", "w") as f:
        f.write(report_text)

# Save as CSV and LaTeX
report_df.to_csv(f"{reports_dir}/classification_report.csv")
report_df.to_latex(f"{reports_dir}/classification_report.tex", float_format="%.4f")

# 2. CONFUSION MATRIX (Overall)
cm = confusion_matrix(y_true, y_pred, labels=labels)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
plt.title(f'Confusion Matrix - {model_alias}')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.savefig(f"{reports_dir}/{model_alias}_confusion_matrix.png")
plt.close()

# 3. CONFUSION MATRIX (Per-Source)
all_source_reports = []  # store metrics for summary
for source in df_analysis['Source'].unique():
    source_df = df_analysis[df_analysis['Source'] == source]
    y_true_src = source_df['True_Label']
    y_pred_src = source_df['Predicted_Label']
    cm_s = confusion_matrix(y_true_src, y_pred_src, labels=labels)

    plt.figure(figsize=(6, 4))
    sns.heatmap(cm_s, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
    plt.title(f'Confusion Matrix - {model_alias} ({source})')
    plt.savefig(f"{reports_dir}/{model_alias}_confusion_matrix_{source}.png")
    plt.close()

    # --- Classification Report ---
    report_dict = classification_report(
        y_true_src, y_pred_src,
        output_dict=True,
        zero_division=0
    )
    report_df = pd.DataFrame(report_dict).transpose().round(4)
    report_df.to_csv(f"{reports_dir}/{model_alias}_classification_report_{source}.csv", index=True)

    # Add macro averages for summary
    all_source_reports.append({
        "source": source,
        "precision": round(report_dict["macro avg"]["precision"], 4),
        "recall": round(report_dict["macro avg"]["recall"], 4),
        "f1_score": round(report_dict["macro avg"]["f1-score"], 4)
        })
# --- Summary Report Across Sources ---
summary_df = pd.DataFrame(all_source_reports)
summary_df.loc["Average"] = summary_df.mean(numeric_only=True)
summary_df.to_csv(f"{reports_dir}/{model_alias}_source_summary_report.csv", index=False)

print("\n✅ Per-source classification reports saved.")
print(f"✅ Summary report saved to: {reports_dir}/{model_alias}_source_summary_report.csv")

# 4. ROC CURVE (One-vs-Rest)
# Convert y_true and probabilities to binarized format for multi-class ROC
y_true_bin = label_binarize(y_true, classes=labels)
y_score = df_analysis[['Prob_Normal', 'Prob_Promo', 'Prob_Smish']].values

plt.figure(figsize=(10, 8))
for i in range(n_classes):
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_score[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f'ROC {labels[i]} (AUC = {roc_auc:.2f})')

plt.plot([0, 1], [0, 1], 'k--', lw=2)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Multi-class ROC Curve (One-vs-Rest)')
plt.legend(loc="lower right")
plt.savefig(f"{reports_dir}/roc_curve_combined.png")
plt.close()

# 5. APPEND TO SUMMARY CSV
summary_data = {
    "Model_Name": model_alias,
    "Accuracy": round(acc_ft, 4),
    "Precision": round(prec_ft, 4),
    "Recall": round(rec_ft, 4),
    "F1": round(f1_ft, 4),
    "Timestamp": pd.Timestamp.now()
}

summary_df = pd.DataFrame([summary_data])
summary_df.to_csv(f"{reports_dir}/summary.csv", index=False)

print(f"✅ Evaluation reports generated in: {reports_dir}")

In [ ]:
import shutil
from google.colab import files
import os

# List of folders to zip and download
folders_to_download = [
    ReportFolderName,
    #"stacking_ensemble_reports"
]

for folder in folders_to_download:
    if os.path.exists(folder):
        zip_filename = f"{folder}.zip"
        # Create zip archive
        shutil.make_archive(folder, 'zip', folder)
        # Download zip
        files.download(zip_filename)
        print(f"✅ Download started for '{zip_filename}'")
    else:
        print(f"⚠️ Folder '{folder}' not found")



In [ ]:
# ============================================
# STEP 10: FINE-TUNED RESULTS SUMMARY
# ============================================
print("\n" + "="*50)
print("📊 FINE-TUNED (LoRA) RESULTS")
print("="*50)

results_df = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision (W)', 'Recall (W)', 'F1-Score (W)'],
    'Fine-Tuned (LoRA)': [acc_ft, prec_ft, rec_ft, f1_ft],
})
print(results_df.round(4).to_markdown(index=False))

print("\n--- Fine-Tuned Confusion Matrix ---")
print(confusion_matrix(y_true, y_pred, labels=VALID_LABELS))
